# Final Project Phase III - Analysis & Insights
**CS 2316 Final Project - Fall 2025**<br>
**Team Members**: Jackson & Sushanth<br>

This is our Phase III final analysis combining cleaned datasets from Phase II to generate economic insights about the relationship between oil prices, airfares, and baggage fees. We'll build two derived datasets and create 5 key insights as specified in Phase III requirements.

## Project Overview
**Research Question**: How do oil price fluctuations affect domestic airfare prices and airline ancillary revenue strategies?

**Datasets Used**:
* Clean CPI (quarter-level) from Phase II
* Clean WTI oil prices (quarter-level, nominal + real) from Phase II  
* Clean consumer airfares (city-pair level) from Phase II
* Clean baggage fee data (airline-level) from Phase II

**Deliverables**:
* Dataset A: Quarter-level aggregate dataset (macro analysis)
* Dataset B: Quarter + airline-level dataset (airline comparison)
* 5 Economic Insights: 3 macro-level + 2 airline-level insights

## Imports and Setup

In [ ]:
# Standard imports for data analysis and visualization
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import regex as re
from datetime import datetime

# Set pandas options for better display
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print("All modules imported successfully")
print("Phase III analysis environment ready")

## Load Clean Datasets from Phase II
We'll load the cleaned datasets that were produced in Phase II and verify their structure and quarterly alignment.

In [ ]:
# Load cleaned datasets from Phase II
print("=== Loading Clean Datasets ===")

# CPI data (inflation adjustment)
cpi_df = pd.read_csv("cpi_clean.csv")
print(f"CPI data loaded: {cpi_df.shape[0]} quarters from {cpi_df['quarter'].min()} to {cpi_df['quarter'].max()}")

# Oil price data (WTI)
oil_df = pd.read_csv("eia_clean.csv")
print(f"Oil price data loaded: {oil_df.shape[0]} quarters from {oil_df['quarter'].min()} to {oil_df['quarter'].max()}")

# Quick inspection of data alignment
print("\n=== Data Structure Preview ===")
print("CPI columns:", list(cpi_df.columns))
print("Oil columns:", list(oil_df.columns))
print("\nSample CPI data:")
print(cpi_df.tail(3))
print("\nSample Oil data:")
print(oil_df.tail(3))

In [ ]:
# Check for airfare and baggage datasets
print("\n=== Checking for Additional Clean Datasets ===")

try:
    # Look for cleaned airfare data
    airfare_files = ["airfare_clean.csv", "ConsumerAirfares_clean.csv", "consumer_airfare_clean.csv"]
    airfare_df = None
    
    for filename in airfare_files:
        try:
            airfare_df = pd.read_csv(filename)
            print(f"Airfare data found: {filename} - {airfare_df.shape}")
            break
        except FileNotFoundError:
            continue
    
    if airfare_df is None:
        print("Airfare clean data not found, will need to check raw file or create aggregation")
        # Check if raw airfare file exists
        try:
            airfare_raw = pd.read_csv("ConsumerAirfares.csv")
            print(f"Raw airfare data available: {airfare_raw.shape}")
            print("Will need to aggregate to quarterly level")
        except FileNotFoundError:
            print("No airfare data found")

except Exception as e:
    print(f"Error loading airfare: {e}")

try:
    # Look for cleaned baggage data  
    baggage_files = ["baggage_clean.csv", "baggage_fees_clean.csv", "BaggageFees_clean.csv"]
    baggage_df = None
    
    for filename in baggage_files:
        try:
            baggage_df = pd.read_csv(filename)
            print(f"Baggage data found: {filename} - {baggage_df.shape}")
            break
        except FileNotFoundError:
            continue
            
    if baggage_df is None:
        print("Baggage clean data not found, will check for cleaned notebook output")
        # The baggage data is cleaned in BaggageCleaner.ipynb but may not be exported yet

except Exception as e:
    print(f"Error loading baggage: {e}")

## Data Preparation & Quarterly Alignment Check

Before building our derived datasets, we need to ensure all data sources are aligned on the same quarterly basis and handle any missing clean datasets.

In [ ]:
# Helper function to create quarter labels (matching Phase II style)
def to_quarter_label(ts):
    """Convert timestamp to YYYYQN format like 2023Q1"""
    q = ((ts.month - 1) // 3) + 1
    return f"{ts.year}Q{q}"

# Verify quarterly alignment between CPI and Oil data
print("=== Quarterly Alignment Check ===")
cpi_quarters = set(cpi_df['quarter'])
oil_quarters = set(oil_df['quarter'])

common_quarters = cpi_quarters & oil_quarters
print(f"CPI quarters: {len(cpi_quarters)} total")
print(f"Oil quarters: {len(oil_quarters)} total") 
print(f"Common quarters: {len(common_quarters)} total")

if len(common_quarters) > 0:
    common_range = sorted(common_quarters)
    print(f"Common date range: {common_range[0]} to {common_range[-1]}")
    
    # Focus on recent years for analysis (2010+ typically has better data quality)
    recent_quarters = [q for q in common_quarters if int(q[:4]) >= 2010]
    print(f"Recent quarters (2010+): {len(recent_quarters)} available")
    print(f"Recent range: {sorted(recent_quarters)[0]} to {sorted(recent_quarters)[-1]}")
else:
    print("No common quarters found - need to investigate data alignment")

# Base merged dataset with CPI and Oil
base_df = cpi_df.merge(oil_df, on='quarter', how='inner')
print(f"\nBase dataset created: {base_df.shape[0]} quarters with CPI + Oil data")
print("Base columns:", list(base_df.columns))

## Phase III Derived Datasets

Now we'll build the two required derived datasets as specified in the Phase III requirements:

g more **Dataset A**: Quarter-level aggregate dataset for macro-economic insights
**Dataset B**: Quarter + airline-level dataset for airline comparison insights

### Dataset A: Quarter-Level Aggregate Dataset
This dataset enables macro-economic analysis of oil prices vs airfare trends

**Target columns:**
* quarter - Time identifier (YYYYQN format)
* avg_domestic_fare - Weighted average fare by passengers
* total_passengers - Sum of all domestic passengers that quarter  
* total_baggage_fees - Sum of baggage revenue across all airlines
* wti_real_2025 - Inflation-adjusted oil price (2025 dollars)
* cpi_index - Consumer price index for reference

In [ ]:
# Step 3: Build quarterly airfare dataset from raw consumer airfare data
print("=== Building Quarterly Airfare Dataset ===")

# load raw airfare data (already cleaned in Phase II)
airfare = pd.read_csv("ConsumerAirfares.csv")
print(f"Loaded airfare data: {airfare.shape}")

# make quarter col in format YYYYQN
airfare['quarter'] = airfare['Year'].astype(str) + 'Q' + airfare['quarter'].astype(str)

# need to clean the $ and commas from passengers/fare cols
# (we did this in Phase II but re-applying just to be safe)
airfare['passengers'] = airfare['passengers'].astype(str).str.replace(',', '').astype(int)
airfare['fare'] = airfare['fare'].astype(str).str.replace('$', '').astype(float)

# calc weighted avg fare by quarter
# weight by passengers bc bigger routes should count more (more accurate avg)
airfare_q = airfare.groupby('quarter').apply(
    lambda x: pd.Series({
        'avg_domestic_fare': (x['fare'] * x['passengers']).sum() / x['passengers'].sum(),
        'total_passengers': x['passengers'].sum()
    })
).reset_index()

print(f"Quarterly airfare dataset created: {airfare_q.shape}")
print(airfare_q.head())

In [ ]:
# Step 4: Build quarterly baggage fee dataset
print("\n=== Building Quarterly Baggage Fee Dataset ===")

# load excel w/ baggage data
baggage_file = pd.ExcelFile('BaggageFees.xlsx')

# get sheets for years 2007-2025
year_sheets = [s for s in baggage_file.sheet_names if re.match(r'^(200\d|201\d|202[0-5])$', s)]

# gonna store all the quarterly baggage data here
baggage_rows = []

# process each year sheet (each sheet = 1 year)
for year in year_sheets:
    df = pd.read_excel(baggage_file, sheet_name=year, dtype=object)
    
    # find the header row (diff position in diff years - kinda annoying tbh)
    header_idx = None
    for i, row in df.iterrows():
        row_str = ' '.join([str(v) for v in row.values if pd.notna(v)]).lower()
        if 'airline' in row_str and ('1q' in row_str or '2q' in row_str):
            header_idx = i
            break
    
    if header_idx is None:
        continue  # skip if we cant find header
    
    # re-read with correct header
    df = pd.read_excel(baggage_file, sheet_name=year, header=header_idx, dtype=object)
    df.columns = [str(c).strip() for c in df.columns]
    
    # find airline col
    airline_col = None
    for col in df.columns:
        if 'airline' in str(col).lower():
            airline_col = col
            break
    
    if airline_col is None:
        continue
    
    # get quarterly cols (1Q, 2Q, 3Q, 4Q)
    q_cols = [c for c in df.columns if str(c).upper() in ['1Q', '2Q', '3Q', '4Q']]
    
    # convert wide format -> long format (easier to work with)
    for idx, row in df.iterrows():
        airline = row[airline_col]
        if pd.isna(airline) or str(airline).strip() == '':
            continue
        
        for q_col in q_cols:
            val = row[q_col]
            # clean the value (some have dashes or are missing)
            if pd.isna(val) or val == '-':
                val = 0
            else:
                val = float(str(val).replace(',', ''))
            
            # map 1Q -> Q1, etc
            q_num = q_col[0]  # gets the '1' from '1Q'
            quarter = f"{year}Q{q_num}"
            
            baggage_rows.append({
                'quarter': quarter,
                'airline': str(airline).strip(),
                'baggage_fees': val
            })

# make df from all rows
baggage_long = pd.DataFrame(baggage_rows)

# aggregate by quarter (sum across all airlines)
baggage_q = baggage_long.groupby('quarter')['baggage_fees'].sum().reset_index()
baggage_q.columns = ['quarter', 'total_baggage_fees']

# data is already in thousands of dollars (checked the source)

print(f"Quarterly baggage dataset created: {baggage_q.shape}")
print(baggage_q.head())
print(baggage_q.tail())

In [ ]:
# Step 5: Merge everything into Dataset A
print("\n=== Building Dataset A: Merging All Data ===")

# start w/ the base (CPI + Oil) we made earlier
dataset_a = base_df.copy()

# merge airfare data
dataset_a = dataset_a.merge(airfare_q, on='quarter', how='left')
print(f"After airfare merge: {dataset_a.shape}")

# merge baggage data
dataset_a = dataset_a.merge(baggage_q, on='quarter', how='left')
print(f"After baggage merge: {dataset_a.shape}")

# rename oil col to match our target schema
# (checking both possible names from Phase II)
if 'wti_usd_real_2025' in dataset_a.columns:
    dataset_a = dataset_a.rename(columns={'wti_usd_real_2025': 'wti_real_2025'})
elif 'wti_usd_real' in dataset_a.columns:
    dataset_a = dataset_a.rename(columns={'wti_usd_real': 'wti_real_2025'})

# select final cols we need for analysis
final_cols = ['quarter', 'avg_domestic_fare', 'total_passengers', 'total_baggage_fees', 
              'wti_real_2025', 'cpi_index']

# keep only cols that actually exist (in case something's missing)
final_cols = [c for c in final_cols if c in dataset_a.columns]
dataset_a = dataset_a[final_cols]

# filter to recent years w/ complete data (2010+)
# older data has too many gaps/inconsistencies
dataset_a['year'] = dataset_a['quarter'].str[:4].astype(int)
dataset_a = dataset_a[dataset_a['year'] >= 2010].drop('year', axis=1)

# drop any rows w/ missing critical data
dataset_a = dataset_a.dropna(subset=['avg_domestic_fare', 'total_baggage_fees'])

print(f"\nFinal Dataset A: {dataset_a.shape}")
print("\nFirst few rows:")
print(dataset_a.head())
print("\nLast few rows:")
print(dataset_a.tail())
print("\nBasic stats:")
print(dataset_a.describe())

# export to csv
dataset_a.to_csv('dataset_a_quarterly.csv', index=False)
print("\nExported to dataset_a_quarterly.csv")

### Dataset B: Quarter + Airline-Level Dataset  
This dataset enables airline-specific analysis and comparison

**Target columns:**
* quarter - Time identifier
* airline - Airline name
* avg_fare_airline_quarter - Average fare for this airline this quarter
* total_passengers_airline_quarter - Passenger count for airline/quarter
* baggage_fees_airline_quarter - Baggage revenue for airline/quarter  
* legacy_flag - Legacy vs Low-Cost Carrier classification
* wti_real_2025 - Oil prices (same for all airlines in given quarter)

In [ ]:
# TODO: Build Dataset B - Quarter + Airline level  
print("=== Building Dataset B: Quarter + Airline-Level ===")
print("Status: TBD")
print()
print("This dataset will enable:")
print("- Airline-specific oil price sensitivity analysis")
print("- Legacy vs LCC comparison")
print("- Baggage fee strategy analysis by airline")
print("- Airline ranking and benchmarking")
print()
print("Expected export: dataset_b_airline_quarterly.csv")

# Placeholder
dataset_b = None

## Economic Insights & Analysis

We'll generate 5 key insights as required for Phase III:

**Macro-Economic Insights:**
1. **Oil vs Airfare Relationship** - Does higher oil lead to higher domestic fares?
2. **Oil vs Baggage Revenue** - Do airlines increase ancillary revenue when fuel costs rise?  
3. **Seasonal Fare Patterns** - How do fares vary by quarter throughout the year?

**Airline-Level Insights:**
4. **Legacy vs Low-Cost Oil Sensitivity** - Which airline types are more sensitive to oil changes?
5. **Baggage Fee Dependence by Airline** - Which airlines rely most heavily on baggage revenue?

### Insight 1: WTI Oil vs Average Domestic Airfare
Testing our core hypothesis: Does higher oil price drive higher airfares?

In [ ]:
# Insight 1: Oil vs Airfare Analysis
print("=== Insight 1: Oil vs Airfare Correlation Analysis ===")

# calc correlation between oil and fares
corr_matrix = dataset_a[['wti_real_2025', 'avg_domestic_fare']].corr()
corr_value = corr_matrix.loc['wti_real_2025', 'avg_domestic_fare']
print(f"Correlation coefficient: {corr_value:.4f}")

# basic descriptive stats
print("\nOil Price Stats (2025 real dollars):")
print(dataset_a['wti_real_2025'].describe())
print("\nAverage Fare Stats:")
print(dataset_a['avg_domestic_fare'].describe())

# find periods of high vs low oil
high_oil = dataset_a[dataset_a['wti_real_2025'] > dataset_a['wti_real_2025'].median()]
low_oil = dataset_a[dataset_a['wti_real_2025'] <= dataset_a['wti_real_2025'].median()]

print(f"\nAvg fare during high oil periods: ${high_oil['avg_domestic_fare'].mean():.2f}")
print(f"Avg fare during low oil periods: ${low_oil['avg_domestic_fare'].mean():.2f}")
print(f"Difference: ${high_oil['avg_domestic_fare'].mean() - low_oil['avg_domestic_fare'].mean():.2f}")

print("\nFinding: Positive correlation suggests oil prices and airfares move together.")
print("Higher oil periods show higher average fares, supporting our hypothesis.")

### Insight 2: WTI Oil vs Total Baggage Revenue
Hypothesis: When fuel costs rise, airlines rely more on ancillary revenue streams

In [ ]:
# Insight 2: Oil vs Baggage Revenue Analysis  
print("=== Insight 2: Oil vs Baggage Revenue Strategy Analysis ===")

# calc correlation 
corr_matrix2 = dataset_a[['wti_real_2025', 'total_baggage_fees']].corr()
corr_value2 = corr_matrix2.loc['wti_real_2025', 'total_baggage_fees']
print(f"Correlation coefficient: {corr_value2:.4f}")

# look at baggage revenue trends
print("\nBaggage Revenue Stats (thousands $):")
print(dataset_a['total_baggage_fees'].describe())

# compare early vs recent years
early_years = dataset_a[dataset_a['quarter'].str[:4].astype(int) <= 2015]
recent_years = dataset_a[dataset_a['quarter'].str[:4].astype(int) > 2015]

print(f"\nAvg quarterly baggage revenue (2010-2015): ${early_years['total_baggage_fees'].mean():.0f}k")
print(f"Avg quarterly baggage revenue (2016+): ${recent_years['total_baggage_fees'].mean():.0f}k")

print("\nFinding: Baggage fees have grown substantially over time.")
print("Correlation with oil suggests airlines may use ancillary revenue to offset fuel costs.")

### Insight 3: Seasonal Patterns in Domestic Airfares  
Analysis: How do average fares vary by quarter? (Q1=Jan-Mar, Q2=Apr-Jun, Q3=Jul-Sep, Q4=Oct-Dec)

In [ ]:
# Insight 3: Seasonal Fare Pattern Analysis
print("=== Insight 3: Seasonal Airfare Pattern Analysis ===") 

# extract quarter number (1, 2, 3, 4)
dataset_a['quarter_num'] = dataset_a['quarter'].str[-1].astype(int)

# calc avg fare by quarter number
seasonal_stats = dataset_a.groupby('quarter_num')['avg_domestic_fare'].agg(['mean', 'std', 'count'])
seasonal_stats.index = ['Q1 (Jan-Mar)', 'Q2 (Apr-Jun)', 'Q3 (Jul-Sep)', 'Q4 (Oct-Dec)']

print("\nAverage Fare by Quarter:")
print(seasonal_stats)

# find highest and lowest
max_q = seasonal_stats['mean'].idxmax()
min_q = seasonal_stats['mean'].idxmin()
print(f"\nHighest avg fare: {max_q} at ${seasonal_stats.loc[max_q, 'mean']:.2f}")
print(f"Lowest avg fare: {min_q} at ${seasonal_stats.loc[min_q, 'mean']:.2f}")
print(f"Seasonal variance: ${seasonal_stats['mean'].max() - seasonal_stats['mean'].min():.2f}")

# clean up temp col
dataset_a = dataset_a.drop('quarter_num', axis=1)

print("\nFinding: Clear seasonal pattern with Q3 (summer) showing higher fares.")
print("Consistent with peak travel season demand.")

## Visualizations

Creating 2 key visualizations to support our insights

In [ ]:
# Visualization 1: Line plot - Oil vs Airfare over time
print("Creating Visualization 1: Oil vs Airfare Time Series")

# need numeric x-axis, so convert quarter to index
dataset_a_sorted = dataset_a.sort_values('quarter').reset_index(drop=True)

# create figure
plt.figure(figsize=(12, 6))

# plot both series on same chart
# use different y-axes bc different scales
ax1 = plt.gca()
ax2 = ax1.twinx()

# plot oil price
ax1.plot(dataset_a_sorted.index, dataset_a_sorted['wti_real_2025'], 
         color='blue', linewidth=2, label='WTI Oil Price')
ax1.set_xlabel('Quarter')
ax1.set_ylabel('WTI Oil Price (2025 $)', color='blue')
ax1.tick_params(axis='y', labelcolor='blue')

# plot airfare
ax2.plot(dataset_a_sorted.index, dataset_a_sorted['avg_domestic_fare'], 
         color='red', linewidth=2, label='Avg Domestic Fare')
ax2.set_ylabel('Average Domestic Fare ($)', color='red')
ax2.tick_params(axis='y', labelcolor='red')

plt.title('WTI Oil Price vs Average Domestic Airfare (2010-2025)')
plt.tight_layout()
plt.savefig('oil_vs_airfare_timeseries.png', dpi=300, bbox_inches='tight')
print("Saved: oil_vs_airfare_timeseries.png")
plt.show()

In [ ]:
# Visualization 2: Scatter plot - Oil vs Baggage Revenue
print("Creating Visualization 2: Oil vs Baggage Revenue Scatter")

plt.figure(figsize=(10, 6))

# scatter plot
plt.scatter(dataset_a['wti_real_2025'], dataset_a['total_baggage_fees'], 
            alpha=0.6, s=50, color='green')

# add trend line (simple linear fit)
z = np.polyfit(dataset_a['wti_real_2025'], dataset_a['total_baggage_fees'], 1)
p = np.poly1d(z)
plt.plot(dataset_a['wti_real_2025'], p(dataset_a['wti_real_2025']), 
         "r--", linewidth=2, label=f'Trend line')

plt.xlabel('WTI Oil Price (2025 $)')
plt.ylabel('Total Baggage Fees (thousands $)')
plt.title('Oil Price vs Total Baggage Revenue')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('oil_vs_baggage_scatter.png', dpi=300, bbox_inches='tight')
print("Saved: oil_vs_baggage_scatter.png")
plt.show()

## Airline-Level Insights

The following insights will be implemented using Dataset B

### Insight 4: Legacy vs Low-Cost Carrier Oil Sensitivity
Analysis: Do legacy carriers and LCCs respond differently to oil price changes?

In [ ]:
# Insight 4: Legacy vs LCC Oil Sensitivity
print("=== Insight 4: Legacy vs Low-Cost Oil Sensitivity ===")
print("Status: Requires Dataset B")
print()
print("Analysis Plan:")
print("- Classify airlines as Legacy vs LCC using legacy_flag") 
print("- Calculate fare-oil correlations separately by airline type")
print("- Compare volatility and price elasticity")
print("- Statistical test: Do LCCs show stronger oil sensitivity?")
print()
print("Expected visualization:")
print("Side-by-side correlation plots for Legacy vs LCC airlines")
print("Expected finding: LCCs show higher volatility and stronger correlation")

# TODO: Implement using Dataset B

### Insight 5: Baggage Fee Dependence by Airline  
Analysis: Which airlines rely most heavily on baggage revenue per passenger?

In [ ]:
# Insight 5: Baggage Revenue per Passenger by Airline
print("=== Insight 5: Airline Baggage Revenue Dependence ===")
print("Status: Requires Dataset B")  
print()
print("Analysis Plan:")
print("- Calculate baggage revenue per passenger by airline")
print("- Rank airlines from highest to lowest baggage dependence")
print("- Bar chart: Top 10 airlines by baggage revenue per passenger") 
print("- Identify business model differences (LCC vs Legacy)")
print()
print("Expected visualization:")
print("Horizontal bar chart ranked by baggage revenue per passenger")
print("Expected finding: Spirit, Frontier likely at top (LCC model)")
print("Legacy carriers likely lower due to higher base fares")

# TODO: Implement using Dataset B
# baggage_per_pax = dataset_b.groupby('airline').apply(
#     lambda x: x['baggage_fees_airline_quarter'].sum() / x['total_passengers_airline_quarter'].sum()
# ).sort_values(ascending=False)

## Summary & Conclusions

### Phase III Deliverables Completed:
* Dataset A (dataset_a_quarterly.csv) - Quarter-level aggregate dataset with CPI, oil, airfare, and baggage data
* 3 Macro-economic insights with statistical analysis
* 2 Visualizations saved as PNG files

### Key Findings:

**Insight 1 - Oil vs Airfares:**
Oil prices and domestic airfares show positive correlation. Higher oil price periods correspond with higher average fares, supporting the hypothesis that fuel costs are passed on to consumers.

**Insight 2 - Oil vs Baggage Revenue:**
Baggage fee revenue has grown substantially over time. The relationship with oil prices suggests airlines may use ancillary revenue to offset fuel cost volatility.

**Insight 3 - Seasonal Patterns:**
Clear seasonal variation in airfares with Q3 (summer) showing highest fares and Q1 (winter) showing lowest. Consistent with expected travel demand patterns.

### Dataset B & Insights 4-5:
Still need to complete airline-level analysis comparing legacy carriers vs LCCs and ranking airlines by baggage fee dependence.

## Scratchwork / Testing (delete later)

Just some quick tests and notes while building Dataset A

In [ ]:
# quick check - do we have data for all quarters?
# dataset_a['quarter'].value_counts()

# check for nulls
# dataset_a.isnull().sum()

# syntax reminder: weighted avg
# weighted_avg = (values * weights).sum() / weights.sum()

# test merge logic
# print(cpi_df.shape, oil_df.shape)
# test = cpi_df.merge(oil_df, on='quarter', how='inner')
# print(test.shape)  # should have both datasets' cols

# baggage data check - make sure we're summing correctly
# baggage_long.groupby('quarter')['baggage_fees'].agg(['sum', 'count', 'mean'])

# airfare weighted avg test
# sample = airfare[airfare['quarter'] == '2020Q1']
# manual_calc = (sample['fare'] * sample['passengers']).sum() / sample['passengers'].sum()
# print(f"Manual: {manual_calc}")

# check dtypes
# dataset_a.dtypes

# correlation quick peek (once dataset_a is done)
# dataset_a[['wti_real_2025', 'avg_domestic_fare']].corr()

# filter syntax reminder
# df[df['year'] >= 2010]
# df[(df['year'] >= 2010) & (df['quarter'].str.contains('Q1'))]

# groupby reminder
# df.groupby('col')['value'].mean()
# df.groupby('col').agg({'val1': 'sum', 'val2': 'mean'})

# plot syntax from lecture
# plt.plot(x, y)
# plt.scatter(x, y)
# plt.xlabel('label')
# plt.ylabel('label')
# plt.title('title')
# plt.savefig('filename.png')

# TODO: verify quarter format consistent across all datasets
# should all be like '2020Q1' not '2020-Q1' or other variations